# Chapter 46: SARSA and DQN Concepts (Pure Python Path)

## Learning Objectives
- Implement SARSA and compare with Q-learning
- Understand on-policy vs off-policy updates
- Build DQN intuition with linear approximation
- Visualize reward curve differences

## Prerequisites
- Chapters 0-45 completed
- Comfort with probability, optimization, and model evaluation
- Optional matplotlib (ASCII fallback provided in scripts)


# Chapter 46: SARSA and DQN Concepts (Pure Python Path)

## Why this chapter matters
After Q-learning, the next step is understanding on-policy learning (SARSA) and function approximation (DQN concepts). This bridges tabular RL to modern deep RL systems.

## Learning goals
1. Implement SARSA update and compare to Q-learning.
2. Understand on-policy vs off-policy behavior.
3. Build intuition for DQN components without heavy frameworks.
4. Visualize learning curves and policy differences.

## SARSA update
\[
Q(s,a) \leftarrow Q(s,a) + \alpha \left[r + \gamma Q(s',a') - Q(s,a)\right]
\]
where `a'` is the next action chosen by the current policy (epsilon-greedy).

## Q-learning vs SARSA
- Q-learning is off-policy: bootstraps with `max_a Q(s',a)`.
- SARSA is on-policy: bootstraps with `Q(s',a')`.
- SARSA is often more conservative under stochastic risk.

## DQN conceptual stack (HF-style reinforcement workflows)
1. Q-network approximation instead of Q-table.
2. Experience replay buffer.
3. Target network for stable bootstrapping.
4. Epsilon schedule for exploration.

## Pure Python DQN intuition exercise
This chapter includes a linear-function approximation toy (not full deep network) to teach:
- TD target
- prediction error
- gradient-like weight updates

## Visualization tasks
1. reward curve by episode
2. success rate over windows
3. SARSA vs Q-learning comparison plot

## Assignment
1. Compare SARSA and Q-learning under slippery transition noise.
2. Add epsilon decay and compare convergence speed.
3. Explain why SARSA can learn safer policy.


In [ ]:
from __future__ import annotations

import random
from typing import Dict, List, Tuple


State = Tuple[int, int]
Action = int


class RiskyGrid:
    """Grid where one middle cell has stochastic negative reward transitions."""

    def __init__(self, size: int = 6, slip_prob: float = 0.15):
        self.size = size
        self.start = (0, 0)
        self.goal = (size - 1, size - 1)
        self.risky = (size // 2, size // 2)
        self.slip_prob = slip_prob
        self.state = self.start

    def reset(self) -> State:
        self.state = self.start
        return self.state

    def step(self, action: Action):
        # 0 up, 1 down, 2 left, 3 right
        r, c = self.state
        if random.random() < self.slip_prob:
            action = random.choice([0, 1, 2, 3])

        if action == 0:
            r -= 1
        elif action == 1:
            r += 1
        elif action == 2:
            c -= 1
        else:
            c += 1

        r = max(0, min(self.size - 1, r))
        c = max(0, min(self.size - 1, c))
        self.state = (r, c)

        done = self.state == self.goal
        reward = 20.0 if done else -0.1
        if self.state == self.risky:
            reward -= 1.0
        return self.state, reward, done


def eps_greedy(Q: Dict[Tuple[State, Action], float], s: State, epsilon: float) -> Action:
    if random.random() < epsilon:
        return random.randint(0, 3)
    return max(range(4), key=lambda a: Q.get((s, a), 0.0))


def train_q_learning(env: RiskyGrid, episodes=1200, alpha=0.15, gamma=0.97, epsilon=0.2):
    Q: Dict[Tuple[State, Action], float] = {}
    rewards = []

    for _ in range(episodes):
        s = env.reset()
        total = 0.0
        for _ in range(250):
            a = eps_greedy(Q, s, epsilon)
            s2, r, done = env.step(a)
            total += r

            best_next = max(Q.get((s2, na), 0.0) for na in range(4))
            old = Q.get((s, a), 0.0)
            Q[(s, a)] = old + alpha * (r + gamma * best_next - old)

            s = s2
            if done:
                break
        rewards.append(total)
    return Q, rewards


def train_sarsa(env: RiskyGrid, episodes=1200, alpha=0.15, gamma=0.97, epsilon=0.2):
    Q: Dict[Tuple[State, Action], float] = {}
    rewards = []

    for _ in range(episodes):
        s = env.reset()
        a = eps_greedy(Q, s, epsilon)
        total = 0.0

        for _ in range(250):
            s2, r, done = env.step(a)
            total += r
            a2 = eps_greedy(Q, s2, epsilon)

            old = Q.get((s, a), 0.0)
            target = r + gamma * Q.get((s2, a2), 0.0)
            Q[(s, a)] = old + alpha * (target - old)

            s, a = s2, a2
            if done:
                break

        rewards.append(total)

    return Q, rewards


def smooth(vals: List[float], w: int = 50) -> List[float]:
    out = []
    for i in range(len(vals)):
        j = max(0, i - w + 1)
        out.append(sum(vals[j : i + 1]) / (i - j + 1))
    return out


def maybe_plot(q_vals: List[float], s_vals: List[float]):
    try:
        import matplotlib.pyplot as plt
    except Exception:
        print("Final avg reward Q-learning:", round(sum(q_vals[-100:]) / 100, 4))
        print("Final avg reward SARSA    :", round(sum(s_vals[-100:]) / 100, 4))
        return

    plt.figure(figsize=(8, 4))
    plt.plot(smooth(q_vals), label="Q-learning")
    plt.plot(smooth(s_vals), label="SARSA")
    plt.title("Reward Curves: Q-learning vs SARSA")
    plt.xlabel("Episode")
    plt.ylabel("Smoothed Reward")
    plt.legend()
    plt.tight_layout()
    plt.show()


def linear_dqn_intuition_step(weights: List[float], features: List[float], target: float, lr: float = 0.05):
    pred = sum(w * x for w, x in zip(weights, features))
    err = pred - target
    for i in range(len(weights)):
        weights[i] -= lr * err * features[i]
    return pred, err


if __name__ == "__main__":
    random.seed(42)
    env = RiskyGrid(size=6, slip_prob=0.18)

    Q_q, rewards_q = train_q_learning(env)
    Q_s, rewards_s = train_sarsa(env)

    print("Q-learning final avg reward:", round(sum(rewards_q[-100:]) / 100, 4))
    print("SARSA final avg reward    :", round(sum(rewards_s[-100:]) / 100, 4))

    maybe_plot(rewards_q, rewards_s)

    # DQN intuition: linear approximation update demo
    w = [0.1, -0.05, 0.02]
    x = [1.0, 0.7, -0.2]
    target_q = 1.4
    for step in range(1, 6):
        pred, err = linear_dqn_intuition_step(w, x, target_q)
        print(f"linear_dqn_step={step} pred={pred:.4f} err={err:.4f} w={[round(v,4) for v in w]}")


## Checkpoint

1. You can derive SARSA update from TD target.
2. You can explain conservative behavior of on-policy learning.
3. You can connect linear Q approximation to DQN ideas.

## Guided Exercise
Run both algorithms over different `slip_prob` values and compare stability.

In [ ]:
# TODO (Student Exercise)
# Sweep slip_prob in [0.0, 0.1, 0.2, 0.3] and compare final rewards for Q-learning vs SARSA.

## Quick Quiz

**Q1.** Why can SARSA learn safer behavior?

**Answer:** Its update uses the next action sampled from exploratory policy.

**Q2.** What makes Q-learning off-policy?

**Answer:** It bootstraps on greedy max next action regardless of behavior action.

**Q3.** Why does DQN need replay buffer?

**Answer:** To decorrelate samples and stabilize optimization.


## Homework
Add epsilon decay and compare sample efficiency for both algorithms.